In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

In [2]:
image_size = (128, 128)
input_shape = (128, 128, 3)
num_classes = 6

In [3]:
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')

])


In [4]:
cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 128, 128, 32)      896       
_________________________________________________________________
batch_normalization (BatchNo (None, 128, 128, 32)      128       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 64, 64, 32)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 64, 64, 64)        18496     
_________________________________________________________________
batch_normalization_1 (Batch (None, 64, 64, 64)        256       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 32, 32, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 32, 32, 128)       7

In [5]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
train_generator = train_datagen.flow_from_directory(
    'newdataset/train',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse'
)

val_generator = val_datagen.flow_from_directory(
    'newdataset/val',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

Found 2527 images belonging to 6 classes.
Found 2527 images belonging to 6 classes.


In [9]:
history = cnn_model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=15,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=[reduce_lr, early_stopping]#checkpoint
)

Epoch 1/15
79/79 [==============================] - 70s 884ms/step - loss: 1.6947 - accuracy: 0.3071 - val_loss: 10.3548 - val_accuracy: 0.1690 - lr: 0.0010
Epoch 2/15
79/79 [==============================] - 51s 642ms/step - loss: 1.6747 - accuracy: 0.3229 - val_loss: 4.6869 - val_accuracy: 0.2465 - lr: 0.0010
Epoch 3/15
79/79 [==============================] - 51s 649ms/step - loss: 1.6717 - accuracy: 0.3190 - val_loss: 4.6113 - val_accuracy: 0.2616 - lr: 0.0010
Epoch 4/15
79/79 [==============================] - 50s 636ms/step - loss: 1.6160 - accuracy: 0.3229 - val_loss: 2.7379 - val_accuracy: 0.3502 - lr: 0.0010
Epoch 5/15
79/79 [==============================] - 51s 641ms/step - loss: 1.5708 - accuracy: 0.3451 - val_loss: 1.5520 - val_accuracy: 0.3953 - lr: 0.0010
Epoch 6/15
79/79 [==============================] - 52s 660ms/step - loss: 1.5762 - accuracy: 0.3498 - val_loss: 1.5513 - val_accuracy: 0.3601 - lr: 0.0010
Epoch 7/15
79/79 [==============================] - 51s 645ms/s

In [7]:
import os
cnn_model.save('transfer_model.h5')

model_path = "transfer_model.h5" 
 
model_size = os.path.getsize(model_path)
 
model_size_mb = model_size / (1024 * 1024)
print(f"Model Size: {model_size_mb:.2f} MB")

Model Size: 16.40 MB


In [8]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

model_path = "transfer_model.h5"
cnn_model = load_model(model_path)

model_size = os.path.getsize(model_path)
model_size_mb = model_size / (1024 * 1024)
print(f"Model Size: {model_size_mb:.2f} MB")

Model Size: 16.40 MB


In [13]:
img_path = "glass_003.jpg"

In [14]:

img = image.load_img(img_path, target_size=(128, 128))
img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = cnn_model.predict(img_array)
predicted_class = np.argmax(pred, axis=1)[0]

In [15]:
class_labels = ['cardboard', 'glass', 'metal', 'paper','plastic','trash'] 
predicted_label = class_labels[predicted_class]

In [16]:

print(f"Predicted Class Index: {predicted_class}")
print(f"Predicted Class Label: {predicted_label}")

Predicted Class Index: 1
Predicted Class Label: glass


In [17]:

xyxy=[]
confidences=[]
class_ids=[]
labels=[]